# 🗺️ Comprehensive Route Analysis for LSE Residences to LSE 🏫

Welcome to this analytical notebook! It is designed to facilitate the London School of Economics (LSE) community's understanding of travel times from various LSE residences to the main campus 🧸. This notebook stands as guidepost of travel insights for students, weaving a detailed narrative of their daily commutes 🛴.

## Objective 🎯
Our objective is to meticulously map out and visualize the journey times for **14 LSE residences** to the LSE campus using multiple modes of transportation: walking 🚶, driving 🚗, bicycling 🚴, bus 🚌, and subway 🚇. Each route's color intensity on the map correlates with the travel duration, providing an intuitive grasp of journey lengths at two critical times:
- Morning rush at 8:00 AM (departure to LSE) 🌅
- Evening return at 6:00 PM (departure from LSE) 🌇

## Methodology 🛠️
For each residence, we:
1. Extract route data for all transport modes using Google Map Directions API.
2. Calculate the duration for the morning and evening commutes, accounting for the expected traffic and transit schedules.
3. Employ `folium` to craft an interactive map, where each route is color-coded. The spectrum ranges from cool to warm hues, denoting travel times from brief to extended. 🎨

## Insights and Impact 💡
Through our visual analysis, students can discern:
- The most time-efficient modes of transportation for each residence.
- Potential time bottlenecks during different times of the day.
- Optimized travel schedules that enhance daily planning and productivity for the LSE family. 📈

🛤️🔍 Let us embark on this journey of discovery! 


### Import Libraries 📚

First we need to import all the libaries we need:

- 🗺️ Plot interactive maps using `folium` to visualize geospatial data. 
- Employ `branca.colormap` to enhance our maps with aesthetically pleasing color schemes 🎨
- Perform comprehensive analysis on our DataFrame to uncover insights🔍
- 🐼 Utilize `pandas` for data manipulation, ensuring our dataset is primed for analysis
- Parse and handle JSON data with the `json` library to work with JSON file formats seamlessly. 📃

In [2]:
import pandas as pd
import json
import folium
from branca.colormap import linear

### ⏰ Duration Exploration

🔍 Here we iterate through each route in our dataset, which is formatted as a JSON Lines file. For each route, we examine all the `legs` and record the `duration` values. Through this, we aim to find the shortest (`min_duration`) and longest (`max_duration`) travel times recorded.

This analysis will help to change the colour of the routes based on the time taken 🚦

In [11]:
file_path = 'demomap/directions_transit_subway_2023-12-24 19:17.jsonl' 

min_duration = float('inf')  
max_duration = float('-inf')

with open(file_path, 'r') as file:
    for line in file:
        data = json.loads(line)
        for route in data.get('routes', []):
            for leg in route.get('legs', []):
                duration = leg['duration']['value'] 
                min_duration = min(min_duration, duration)
                max_duration = max(max_duration, duration)

print(f'Minimum duration: {min_duration}')
print(f'Maximum duration: {max_duration}')


Minimum duration: 743
Maximum duration: 1945


### Color Mapping 🎨

We open the JSON Lines file containing the route information and iterate over each line. Each line is a JSON string representing a single journey, which we decode into a Python dictionary using json.loads and collect into our all_routes list 🔄

🌈 Moving forward, we use branca.colormap to create a spectrum of colors that correspond to the journey durations

The print statement demonstrates how a duration of 1000 seconds is mapped to a color in our gradient.

In [12]:
all_routes = []
with open(file_path, 'r') as file:
    for line in file:
        all_routes.append(json.loads(line))

colormap = linear.YlOrRd_09.scale(min_duration, max_duration).to_step(n=10)
print(colormap(1000)) 

#ffde7fff


### ✨ FUNCTION ✨ - Plotting routes by mode

Initializing the Map: We begin by creating a folium map centered on London's coordinates. This map serves as our canvas 📜

Iterating Over Routes: For each route in our dataset, we further iterate through its legs 🦵 Each leg represents a segment of the journey.

🖍️ Duration and Color Mapping: We extract the duration of each leg and use our previously defined colormap to determine its color. This color reflects the time taken, offering an intuitive grasp of duration.

Marking Start and End Points: We mark the start and end points of each leg with green and red markers, respectively. These markers provide clear visual cues for the journey's beginning and end 📍

🖌️ Drawing the Route: We plot the route on the map using folium.PolyLine, which takes a list of coordinates (latitude and longitude) for each step of the leg. The line's color and opacity are adjusted according to the journey's duration. 

Returning the Map: After plotting all routes and legs, the function returns the folium map object, now enriched with our route data.

This function is vital for visualizing the various routes from LSE residences to the campus, allowing us to appreciate the spatial and temporal aspects of each journey 👩‍🎨

In [13]:
def plot_all_routes_by_mode_combined(routes, mode):
    
    map_routes = folium.Map(location=[51.5074, -0.1278], zoom_start=12)
    
    for route_data in routes:
        for leg in route_data['routes'][0]['legs']:

            duration = leg['duration']['value']
            # normalized_duration = duration / (max_duration - min_duration)
            color = colormap(duration)
            # print(f"Duration: {duration}, Color: {color}")
           
            start_coords = [leg['start_location']['lat'], leg['start_location']['lng']]
            end_coords = [leg['end_location']['lat'], leg['end_location']['lng']]

            
            folium.Marker(start_coords, icon=folium.Icon(color='green')).add_to(map_routes)
            folium.Marker(end_coords, icon=folium.Icon(color='red')).add_to(map_routes)

            steps = leg['steps']
            coords = [(step['start_location']['lat'], step['start_location']['lng']) for step in steps]
            folium.PolyLine(coords, color='black',weight=8, opacity=1).add_to(map_routes)
            folium.PolyLine(coords, color=color, weight=5, opacity=1).add_to(map_routes)

    return map_routes

### Creating HTML files 📈

🔄 Looping Through Modes:

We have a list modes_combined that includes the different modes of transportation we're interested in: walking 🚶, driving 🚗, bicycling 🚴, bus 🚌, and subway 🚇.

✅ For each mode, we call our function plot_all_routes_by_mode_combined, which creates a folium map visualizing the routes for that specific mode.

Saving and Organizing Maps 💾 :

Each map is saved as an HTML file, named distinctively based on the transportation mode (updated_map_all_{mode}.html). This allows for easy identification and access later 🌊

The file paths for these maps are stored in a dictionary file_paths_all_routes_combined, keyed by the mode of transportation. This organization provides a structured way to access the maps for each transportation mode.

🗺️ End Result:

We end up with a collection of interactive maps, each representing a different way to traverse the routes from LSE residences to the campus 😎

The saved HTML files can be conveniently opened in any web browser, offering an engaging and informative view of the journey options available to the LSE community.

In [10]:
coloured_map = plot_all_routes_by_mode_combined(all_routes, 'subway')
file_name = f'map_subway.html'
coloured_map.save(file_name)

### 🔬 Simplify Data Structure for further analysis

The flatten_json ✨ FUNCTION ✨ is designed to "flatten" nested JSON objects, converting them into a simple key-value pair format. Each key in the output dictionary represents a path through the original nested structure, leading to a specific value.

🧑‍🔬 Recursive Approach: It employs a recursive strategy, where the function calls itself to handle deeper levels of nesting. 

Handling Dictionaries: When encountering a dictionary, it iterates over each key-value pair, appending the key to a running string that represents the path in the nested structure 🪺

🎣 Handling Lists: For lists, it iterates over each element, using the index as part of the path.

Termination: When it reaches a non-dict and non-list value, it adds the path-value pair to the output dictionary 🏁

💡  By flattening the JSON data, we transform it into a structure that can be easily analyzed using tools like pandas DataFrames. This method makes it easier to access specific data points and conduct thorough statistical analysis or visualization. It streamlines the process of extracting meaningful insights from complex JSON files, enhancing the efficiency of our data processing workflow.

In [25]:
def flatten_json(y):
    out = {}    
    def flatten(x, name=''):
        if type(x) is dict:
            for a in x:
                flatten(x[a], f'{name}{a}_')
        elif type(x) is list:
            for i, a in enumerate(x):
                flatten(a, f'{name}{i}_')
        else:
            out[name[:-1]] = x
    flatten(y)
    return out